# GSS Data Processing

This notebook creates a subset of the General Social Survey (GSS) data for analysis.
It extracts selected variables from the 2022 survey year.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

In [2]:
DATA_DIR = Path("data")
INPUT_FILE = DATA_DIR / "GSS_stata" / "gss7224_r2.parquet"
OUTPUT_FILE = DATA_DIR / "gss_2022.csv"

YEAR = 2022

VARIABLES = [
    "year",
    "id",
    "age",
    "sex",
    "race",
    "degree",
    "satjob",
    "hlthdep",
    "feeldown",
    "nointerest",
    "feelnerv",
    "worry",
    "wrkmeangfl",
    "richwork",
    "satfin",
    "discaffwnv",
    "lifenow",
]

In [3]:
gss_full = pd.read_parquet(INPUT_FILE)
print(f"Full dataset: {gss_full.shape[0]:,} rows, {gss_full.shape[1]} columns")

Full dataset: 75,699 rows, 6904 columns


In [4]:
gss_subset = gss_full.query("year == @YEAR")[VARIABLES].copy()
print(f"Subset: {gss_subset.shape[0]:,} rows, {gss_subset.shape[1]} columns")

Subset: 3,544 rows, 17 columns


In [5]:
gss_subset.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3544 entries, 68846 to 72389
Data columns (total 17 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   year        3544 non-null   int16  
 1   id          3544 non-null   int16  
 2   age         3336 non-null   float64
 3   sex         3524 non-null   float64
 4   race        3491 non-null   float64
 5   degree      3544 non-null   float64
 6   satjob      2463 non-null   float64
 7   hlthdep     1127 non-null   float64
 8   feeldown    1942 non-null   float64
 9   nointerest  1942 non-null   float64
 10  feelnerv    1942 non-null   float64
 11  worry       1939 non-null   float64
 12  wrkmeangfl  1953 non-null   float64
 13  richwork    1452 non-null   float64
 14  satfin      3526 non-null   float64
 15  discaffwnv  590 non-null    float64
 16  lifenow     1780 non-null   float64
dtypes: float64(15), int16(2)
memory usage: 456.8 KB


In [6]:
gss_subset.describe()

,year,id,age,sex,race,degree,satjob,hlthdep,feeldown,nointerest,feelnerv,worry,wrkmeangfl,richwork,satfin,discaffwnv,lifenow
count,3544.0,3544.000000,3336.000000,3524.000000,3491.000000,3544.000000,2463.000000,1127.000000,1942.000000,1942.000000,1942.000000,1939.000000,1953.000000,1452.00000,3526.000000,590.000000,1780.000000
mean,2022.0,1772.535553,49.177458,1.538309,1.397880,1.807844,1.736906,2.244898,1.467044,1.435118,1.709063,1.635895,1.801331,1.34573,2.057856,2.123729,7.882584
std,0.0,1023.268418,17.973585,0.498601,0.689742,1.257677,0.806532,1.185469,0.770860,0.743623,0.889176,0.891116,0.716076,0.47577,0.743043,0.866239,1.548987
min,2022.0,1.000000,18.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.000000,1.000000
25%,2022.0,886.750000,34.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000,2.000000,1.250000,7.000000
50%,2022.0,1772.500000,48.000000,2.000000,1.000000,1.000000,2.000000,2.000000,1.000000,1.000000,1.000000,1.000000,2.000000,1.00000,2.000000,2.000000,8.000000
75%,2022.0,2658.250000,64.000000,2.000000,2.000000,3.000000,2.000000,3.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.00000,3.000000,3.000000,9.000000
max,2022.0,3545.000000,89.000000,2.000000,3.000000,4.000000,4.000000,5.000000,4.000000,4.000000,4.000000,4.000000,4.000000,2.00000,3.000000,4.000000,10.000000


In [12]:
missing = gss_subset.isna().sum()
missing_pct = (missing / len(gss_subset) * 100).round(1)
missing_df = pd.DataFrame({"Missing": missing, "% Missing": missing_pct})
print("Missing values per variable:")
missing_df.sort_values(by="% Missing", ascending=False)

Missing values per variable:


,Missing,% Missing
discaffwnv,2954,83.4
hlthdep,2417,68.2
richwork,2092,59.0
lifenow,1764,49.8
worry,1605,45.3
feelnerv,1602,45.2
feeldown,1602,45.2
nointerest,1602,45.2
wrkmeangfl,1591,44.9
satjob,1081,30.5


## Exploring FEELNERV and WORRY

These two variables measure related anxiety constructs and may be candidates for combining.

In [8]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("FEELNERV", "WORRY"))

feelnerv_counts = gss_subset["feelnerv"].value_counts().sort_index()
worry_counts = gss_subset["worry"].value_counts().sort_index()

fig.add_trace(
    go.Bar(x=feelnerv_counts.index, y=feelnerv_counts.values, name="FEELNERV"),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=worry_counts.index, y=worry_counts.values, name="WORRY"),
    row=1, col=2
)

fig.update_layout(
    title_text="Distribution of FEELNERV and WORRY",
    showlegend=False,
    height=400
)
fig.update_xaxes(title_text="Response", row=1, col=1)
fig.update_xaxes(title_text="Response", row=1, col=2)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.show()

In [9]:
crosstab = pd.crosstab(gss_subset["feelnerv"], gss_subset["worry"])

fig = px.imshow(
    crosstab,
    labels=dict(x="WORRY", y="FEELNERV", color="Count"),
    title="Joint Distribution of FEELNERV and WORRY",
    text_auto=True,
    color_continuous_scale="Blues"
)
fig.update_layout(height=500, width=600)
fig.show()

In [10]:
from scipy.stats import spearmanr, kendalltau

complete_cases = gss_subset[["feelnerv", "worry"]].dropna()
n = len(complete_cases)

pearson_corr = complete_cases["feelnerv"].corr(complete_cases["worry"])
spearman_corr, spearman_p = spearmanr(complete_cases["feelnerv"], complete_cases["worry"])
kendall_corr, kendall_p = kendalltau(complete_cases["feelnerv"], complete_cases["worry"])

print(f"Correlation between FEELNERV and WORRY (n={n:,}):")
print(f"  Pearson r:  {pearson_corr:.3f}")
print(f"  Spearman ρ: {spearman_corr:.3f} (p={spearman_p:.2e})")
print(f"  Kendall τ:  {kendall_corr:.3f} (p={kendall_p:.2e})")

Correlation between FEELNERV and WORRY (n=1,933):
  Pearson r:  0.707
  Spearman ρ: 0.692 (p=4.65e-275)
  Kendall τ:  0.656 (p=2.15e-223)


In [11]:
gss_subset.to_csv(OUTPUT_FILE, index=False)
print(f"Saved to {OUTPUT_FILE}")

Saved to data/gss_2022.csv
